In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = np.pad(x_train.astype('float32') / 255.0,
                 ((0,0),(2,2),(2,2)), mode='constant')
x_test  = np.pad(x_test.astype('float32') / 255.0,
                 ((0,0),(2,2),(2,2)), mode='constant')
x_train = x_train[..., None]
x_test  = x_test[..., None]
y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

model = keras.Sequential([
    layers.Conv2D(6, (5,5), activation='relu', input_shape=(32,32,1), name='conv1'),
    layers.AveragePooling2D((2,2), strides=2, name='pool1'),
    layers.Conv2D(16, (5,5), activation='relu', name='conv2'),
    layers.AveragePooling2D((2,2), strides=2, name='pool2'),
    layers.Conv2D(120, (5,5), activation='relu', name='conv3'),
    layers.Flatten(),
    layers.Dense(84, activation='relu', name='dense1'),
    layers.Dense(10, activation='softmax', name='output')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test))

def exportar_float(arr, nome):
    """Cria header C com array float32"""
    flat = arr.flatten()
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"// Dimensoes: {list(arr.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const float {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(f"{v:.8f}f" for v in flat[i:i+8]) for i in range(0, len(flat), 8)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def exportar_uint8(arr, nome):
    """Cria header C com array uint8_t"""
    flat = arr.flatten().astype(np.uint8)
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"#include <stdint.h>\n\n"
    cabecalho += f"// Dimensoes: {list(arr.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const uint8_t {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(str(v) for v in flat[i:i+16]) for i in range(0, len(flat), 16)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

for camada in ['conv1','conv2','conv3','dense1','output']:
    w, b = model.get_layer(camada).get_weights()
    with open(f"{camada}_w_float.h", "w") as f:
        f.write(exportar_float(w, f"{camada}_w_float"))
    with open(f"{camada}_b_float.h", "w") as f:
        f.write(exportar_float(b, f"{camada}_b_float"))

img_teste = (x_test[0] * 255.0).astype(np.uint8)
with open("imagem_teste_uint8.h", "w") as f:
    f.write(exportar_uint8(img_teste, "imagem_teste_uint8"))

print("Arquivos gerados com sucesso.")

!zip pesos_lenet5.zip *_float.h imagem_teste_uint8.h
from google.colab import files
files.download('pesos_lenet5.zip')

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (AveragePooling2D)        │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (AveragePooling2D)        │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 22s 43ms/step - accuracy: 0.8876 - loss: 0.3838 - val_accuracy: 0.9624 - val_loss: 0.1224
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 44ms/step - accuracy: 0.9675 - loss: 0.1067 - val_accuracy: 0.9768 - val_loss: 0.0719
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.9776 - loss: 0.0721 - val_accuracy: 0.9808 - val_loss: 0.0606
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 40ms/step - accuracy: 0.9818 - loss: 0.0581 - val_accuracy: 0.9868 - val_loss: 0.0478
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.9846 - loss: 0.0487 - val_accuracy: 0.9842 - val_loss: 0.0496
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 41ms/step - accuracy: 0.9871 - loss: 0.0421 - val_accuracy: 0.9897 - val_loss: 0.0348
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.9888 - loss: 0.0359 - val_accuracy: 0.9876 - val_loss: 0.0397
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.9896 - loss: 0.0323 - 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pred = model.predict(x_test[0:1])
print(pred)
print("Classe prevista:", np.argmax(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
[[1.2655211e-08 2.1785223e-07 4.6320713e-07 1.9978334e-06 1.5949610e-08
  3.7908938e-09 6.4771371e-13 9.9999535e-01 3.1966717e-08 1.8955284e-06]]
Classe prevista: 7
